# Healthcare Data Integration and Analysis

## Overview

Healthcare organizations often store patient data across disconnected systems — EHRs, pharmacy platforms, and clinical labs. This notebook walks through the full pipeline I built to **unify, clean, and analyze** that data to surface population health insights.

**Three data sources:**

| File | System | Records |
|---|---|---|
| `Patients_EHR_raw_data.csv` | Electronic Health Record | 74 patients |
| `Pharmacy_Claims.txt` | Pharmacy | 100 claims |
| `Lab_Results.txt` | Clinical Lab | 201 test results |

**What this notebook covers:**

1. Converting raw text files into spreadsheet format
2. Analyzing pharmacy billing and lab diagnostic data
3. Auditing and cleaning the EHR dataset
4. Querying across all three sources using SQL


---
## Setup

In [ ]:
import sqlite3
import pandas as pd
import re
from pathlib import Path
from datetime import datetime, timedelta

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
print("Ready")

---
## 1. Importing the Data

The pharmacy and lab files were delivered as plain-text CSV files. I converted them to `.xlsx` so they can be opened in Excel and shared with clinical stakeholders.


### Pharmacy Claims

In [ ]:
pc = pd.read_csv("data/raw/Pharmacy_Claims.txt")
pc.to_excel("data/processed/Pharmacy_Claims.xlsx", index=False)

print(f"Pharmacy Claims — {pc.shape[0]} rows × {pc.shape[1]} columns")
pc.head()

### Lab Results

In [ ]:
lr = pd.read_csv("data/raw/Lab_Results.txt")
lr.to_excel("data/processed/Lab_Results.xlsx", index=False)

print(f"Lab Results — {lr.shape[0]} rows × {lr.shape[1]} columns")
lr.head()

---
## 2. Analyzing Claims and Lab Data

### 2.1 Pharmacy Claims — Billing & Payer Analysis


#### How many claims had a paid amount over $250?

In [ ]:
high_paid = pc[pc['PaidAmount'] > 250]
print(f"Claims with PaidAmount > $250: {len(high_paid)}")
high_paid[['ClaimID','PatientID','Medication','Payer','PaidAmount']].sort_values('PaidAmount', ascending=False)

#### Which payer reimbursed the most in total?

In [ ]:
by_payer = pc.groupby('Payer')['PaidAmount'].sum().sort_values(ascending=False)
print("Total reimbursed by payer:")
print(by_payer.to_string())
print(f"\nTop payer: {by_payer.idxmax()} — ${by_payer.max():,.2f}")

### 2.2 Lab Results — LDL Cholesterol Analysis

#### What is the highest recorded LDL Cholesterol value?

In [ ]:
ldl = lr[lr['TestName'] == 'LDL Cholesterol'].copy()
print(f"Max LDL Cholesterol: {ldl['TestResultValue'].max()} mg/dL")
ldl.sort_values('TestResultValue', ascending=False).head(10)

#### What is the average LDL Cholesterol?

In [ ]:
avg = ldl['TestResultValue'].mean()
print(f"Average LDL Cholesterol: {avg:.2f} mg/dL")

#### How many patients have dangerously high LDL (> 160 mg/dL)?

In [ ]:
high_risk = ldl[ldl['TestResultValue'] > 160]
print(f"Patients with LDL > 160 mg/dL: {high_risk['PatientID'].nunique()}")
high_risk[['PatientID','TestResultValue','CollectionDate','AbnormalFlag']].sort_values('TestResultValue', ascending=False)

---
## 3. EHR Data Quality Audit and Cleaning

Before using the EHR data in any joins or analysis, I did a full quality audit. A dataset of 74 patients had 15 separate data issues — roughly a 10% problem rate.


In [ ]:
df = pd.read_csv("data/raw/Patients_EHR_raw_data.csv", encoding="utf-8-sig")
print(f"Shape: {df.shape}")
df.head()

### 3.1 Finding the Issues

#### Missing values

In [ ]:
missing = df.isnull().sum()
print(missing)
print(f"\nTotal missing: {missing.sum()}")

#### Duplicate rows

In [ ]:
print(f"Duplicate rows: {df.duplicated().sum()}")
df[df.duplicated(keep=False)]

#### Inconsistent Gender encoding

In [ ]:
print(df['Gender'].value_counts())
print(f"\nNon-standard values: {list(df[~df['Gender'].isin(['Male','Female'])]['Gender'].unique())}")

#### Mixed date formats in AdmissionDate

In [ ]:
def classify_date(val):
    v = str(val).strip()
    if re.match(r'^\d{2}-\d{2}-\d{4}$', v): return 'DD-MM-YYYY'
    if re.match(r'^\d{4}-\d{2}-\d{2}$', v): return 'YYYY-MM-DD'
    try: float(v); return 'Excel serial number'
    except: return 'Unknown'

df['_fmt'] = df['AdmissionDate'].apply(classify_date)
print(df['_fmt'].value_counts())

print("\nRows with non-standard dates:")
display(df[df['_fmt'] != 'DD-MM-YYYY'][['PatientID','AdmissionDate','_fmt']])
df.drop(columns=['_fmt'], inplace=True)

### 3.2 Fixing the Issues

#### Fill missing values

In [ ]:
# Categorical columns → fill with 'Unknown'
df['DateOfBirth'] = df['DateOfBirth'].fillna('Unknown')
df['ZipCode']     = df['ZipCode'].fillna('Unknown')

print("Missing values remaining:", df.isnull().sum().sum())

#### Remove duplicate rows

In [ ]:
before = len(df)
df = df.drop_duplicates(keep='first')
print(f"Removed {before - len(df)} duplicate row(s). {len(df)} rows remain.")

#### Standardize Gender

In [ ]:
df['Gender'] = df['Gender'].replace({
    'M': 'Male', 'm': 'Male', 'male': 'Male',
    'F': 'Female', 'f': 'Female'
})
print(df['Gender'].value_counts())

#### Standardize date formats

Two dates were stored as Excel serial integers — `45258` and `45439`. This happens when a date cell in Excel is saved as a number without proper formatting. I converted these back to real dates and standardized everything to `YYYY-MM-DD`.


In [ ]:
def parse_date(val):
    v = str(val).strip()
    for fmt in ['%d-%m-%Y', '%Y-%m-%d']:
        try: return datetime.strptime(v, fmt).strftime('%Y-%m-%d')
        except: pass
    try:
        # Excel serial number (days since 1899-12-30)
        return (datetime(1899, 12, 30) + timedelta(days=int(float(v)))).strftime('%Y-%m-%d')
    except: return v

df['AdmissionDate'] = df['AdmissionDate'].apply(parse_date)
df['AdmissionDate'] = pd.to_datetime(
    df['AdmissionDate'], infer_datetime_format=True, errors='coerce'
).dt.strftime('%Y-%m-%d')

all_ok = df['AdmissionDate'].str.match(r'^\d{4}-\d{2}-\d{2}$').all()
print(f"All dates in YYYY-MM-DD: {all_ok}")
df[['PatientID','AdmissionDate']].head(10)

#### Save the cleaned dataset

In [ ]:
out = "data/processed/Patients_EHR_cleaned.csv"
df.to_csv(out, index=False)
print(f"Saved: {out}")
print(f"Final shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

---
## 4. SQL Cross-System Analysis

With all three datasets clean, I loaded them into a local SQLite database and wrote queries that join across the systems — something that's impossible when the data lives in separate files.


In [ ]:
db_path = Path("data/processed/medical_data.db")

def _std(val):
    v = str(val).strip()
    for fmt in ['%d-%m-%Y','%Y-%m-%d']:
        try: return datetime.strptime(v, fmt).strftime('%Y-%m-%d')
        except: pass
    try: return (datetime(1899,12,30)+timedelta(days=int(float(v)))).strftime('%Y-%m-%d')
    except: return v

# Load and clean patients for the DB
ehr = pd.read_csv("data/raw/Patients_EHR_raw_data.csv", encoding="utf-8-sig")
ehr['DateOfBirth']   = ehr['DateOfBirth'].fillna('Unknown')
ehr['ZipCode']       = ehr['ZipCode'].fillna('Unknown')
ehr.drop_duplicates(inplace=True)
ehr['Gender']        = ehr['Gender'].replace({'M':'Male','m':'Male','male':'Male','F':'Female','f':'Female'})
ehr['AdmissionDate'] = ehr['AdmissionDate'].apply(_std)
ehr['DateOfBirth']   = ehr['DateOfBirth'].apply(lambda x: _std(x) if x != 'Unknown' else x)

conn = sqlite3.connect(db_path)
ehr.to_sql('patients',        conn, if_exists='replace', index=False)
pc.to_sql('pharmacy_claims',  conn, if_exists='replace', index=False)
lr.to_sql('lab_results',      conn, if_exists='replace', index=False)
conn.commit()

def q(sql): return pd.read_sql_query(sql, conn)

print(f"Database ready: {db_path}")

### Patients born before 1980

In [ ]:
q("""
    SELECT PatientID, Gender, DateOfBirth
    FROM   patients
    WHERE  DateOfBirth < '1980-01-01'
      AND  DateOfBirth != 'Unknown'
    ORDER  BY DateOfBirth;
""")

### Female patients with Type 2 Diabetes

In [ ]:
q("""
    SELECT PatientID, Gender, DateOfBirth, PrimaryCondition
    FROM   patients
    WHERE  Gender = 'Female'
      AND  PrimaryCondition LIKE '%Type 2 Diabetes%'
    ORDER  BY PatientID;
""")

### Lab results joined with patient demographics

In [ ]:
q("""
    SELECT p.PatientID, p.Gender, p.DateOfBirth,
           l.TestName, l.TestResultValue, l.Units, l.AbnormalFlag
    FROM   patients    p
    JOIN   lab_results l ON p.PatientID = l.PatientID
    ORDER  BY p.PatientID, l.TestName;
""")

### Patients and their insurance payer(s)

In [ ]:
q("""
    SELECT DISTINCT p.PatientID, p.Gender, p.DateOfBirth, pc.Payer
    FROM   patients        p
    JOIN   pharmacy_claims pc ON p.PatientID = pc.PatientID
    ORDER  BY p.PatientID;
""")

In [ ]:
conn.close()

---
## Summary

| What I did | Result |
|---|---|
| Converted pharmacy & lab files to Excel | 2 `.xlsx` files |
| Claims analyzed for billing patterns | 21 claims > $250 paid |
| Top payer identified | Private — $7,313.09 total |
| Max LDL Cholesterol found | 187.1 mg/dL |
| Average LDL Cholesterol | 134.64 mg/dL |
| High-risk patients (LDL > 160) | 12 patients |
| EHR records audited | 74 rows, 15 issues found |
| EHR after cleaning | 72 rows, 0 issues |
| SQL queries across 3 systems | 4 queries, 180+ joined records |

### What stood out

- **Private insurers** pay the most in total — useful for contract and budget planning.
- **12 patients** have LDL above 160 mg/dL and should be flagged for clinical follow-up.
- **~10% of EHR records** had at least one quality issue, highlighting the need for validation at data entry.
- **Excel serial number dates** are easy to overlook — both affected rows would have silently broken any date-based query without this audit.
